# Version v3 sans raccourci de format — entraînement et comparaison sur la validation (notebook 12)

**Pourquoi cette version ?** Le notebook 11 a montré que :

- le **format de l'image d'origine** suffit presque à séparer les classes (AUC 0,935 sur la validation, sans aucun pixel) : les radios PNEUMONIA sont plus petites et plus larges ;
- avec la préparation v2, ce format se voit dans l'épaisseur des **bandes noires** ajoutées au redimensionnement, et les modèles placent 53 à 59 % de leur carte de décision « pneumonie » dans ces bandes ;
- les **statistiques globales d'intensité** séparent aussi les classes (AUC 0,908) : les radios NORMAL ont plus de contraste.

**Seule la préparation des images change :**

| Étape | v2 | v3 |
|---|---|---|
| Redimensionnement | proportions conservées, bandes noires ajoutées | image étirée en 224 × 224, sans bandes noires |
| Intensité | pixels ÷ 255 | égalisation d'histogramme de chaque image (`ImageOps.equalize`), puis ÷ 255 |
| Augmentation | aucune, ou luminosité et contraste | aucune : l'égalisation rend déjà le modèle insensible aux changements globaux de luminosité et de contraste |

**Ce qui ne change pas**, pour que la comparaison isole l'effet de la préparation : les architectures (CNN classique, CNN + CBAM résiduel), les réglages retenus (classique : LR 0,001 et Dropout 0,5 ; CBAM : LR 0,0005 et Dropout 0,3), les graines 42, 123 et 2026, l'optimiseur, les poids des classes et le budget (20 époques au plus, patience 4 sur le F1 macro de validation). À graine égale, les poids initiaux et l'ordre des lots sont identiques à ceux des modèles v2 sans augmentation : seule l'image d'entrée diffère.

**Limites fixées à l'avance :**

- Les réglages ont été choisis avec la préparation v2 et ne sont pas réoptimisés pour v3, faute de temps de calcul. Cela peut désavantager un peu v3.
- Des indices résiduels peuvent subsister : l'étirement déforme davantage les images larges ; la résolution d'origine (les NORMAL sont plus grandes) peut laisser une trace dans la texture ; les marqueurs « R » restent visibles. Le notebook mesure la netteté des entrées et la dépendance au format pour les quantifier.
- Une **baisse du F1 de validation est attendue** : une partie des performances v2 venait de raccourcis également présents dans la validation.

## Hypothèses pour l'évaluation finale sur le test, fixées le 28/09/2026, avant l'entraînement v3

Le test n'est pas chargé dans ce notebook. Ces hypothèses seront évaluées une seule fois, sur les 18 modèles (v2 sans augmentation, v2 avec augmentation, v3 ; 2 architectures ; 3 graines) :

1. **Question du mémoire :** pour chaque version, écart de F1 macro CBAM − classique, apparié par graine. Aucun sens n'est prédit.
2. **Raccourci :** la spécificité NORMAL des modèles v3 est supérieure à celle des modèles v2 sans augmentation, à architecture et graine égales.
3. **Augmentation :** les modèles v2 avec augmentation font moins de faux positifs que les modèles v2 sans augmentation, à architecture et graine égales.
4. **Mécanisme :** pour les modèles v2, la probabilité de classer une radio NORMAL en PNEUMONIA augmente avec le rapport largeur/hauteur de l'image d'origine ; cette association est plus faible pour v3.

Mesure principale : F1 macro. Mesures secondaires : AUC, rappel PNEUMONIA, spécificité NORMAL. Seuil de décision : classe du plus grand score, sans ajustement. Incertitude : rééchantillonnage apparié par patient (bootstrap par groupe, 2 000 tirages) et dispersion entre les 3 graines.

**Exécution :** noyau `memoire_cnn`, cellules dans l'ordre. Préparation des 4 753 images : quelques minutes. Entraînement : 6 modèles, environ 3 à 6 heures sur CPU. Les images préparées restent en mémoire (environ 240 Mo) : elles ne sont pas relues à chaque époque. Après une interruption, réexécuter toutes les cellules : chaque entraînement reprend à la dernière époque complète et les entraînements terminés sont reconnus. **Pendant l'entraînement, garder OneDrive en pause, fermer les CSV du projet dans Excel et empêcher la mise en veille.**

In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import sys
import time
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display

# Si le projet change de dossier, modifier uniquement PROJET.
PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
DOSSIER_08 = PROJET / "optimisation" / "robustesse_validation_v2"
DOSSIER_10 = PROJET / "optimisation" / "robustesse_augmentation_v2"
RACINE = PROJET / "optimisation" / "v3_sans_raccourci_format"
APPAREIL = torch.device("cpu")
TAILLE, LOT, MAX_EPOQUES, PATIENCE = 224, 16, 20, 4
GRAINES = [42, 123, 2026]
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
PRETRAITEMENT_V3 = "L; resize 224x224 bilineaire sans padding (etirement); ImageOps.equalize; /255; aucune augmentation"
VERSIONS_MODELES = ["v2 sans augmentation", "v2 avec augmentation", "v3 sans raccourci"]
# Palette catégorielle validée (daltonisme) : une couleur par version, dans cet ordre fixe.
COULEURS = {"v2 sans augmentation": "#2a78d6", "v2 avec augmentation": "#eb6834", "v3 sans raccourci": "#1baf7a"}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def empreinte_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def remplacer(temporaire, chemin, essais=20, attente=1.5):
    # Windows ou OneDrive peuvent verrouiller un fichier quelques secondes : on réessaie.
    for essai in range(essais):
        try:
            temporaire.replace(chemin)
            return
        except PermissionError:
            if essai == essais - 1:
                raise PermissionError(f"Impossible d'écrire {chemin} : fermer ce fichier (Excel ?) "
                                      "ou suspendre OneDrive, puis relancer la cellule.")
            time.sleep(attente)

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    remplacer(temporaire, chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    remplacer(temporaire, chemin)

def sauver_torch(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    torch.save(objet, temporaire)
    remplacer(temporaire, chemin)

def enregistrer_figure(figure, chemin):
    temporaire = chemin.with_suffix(".tmp")
    figure.savefig(temporaire, format="png", dpi=160, bbox_inches="tight")
    remplacer(temporaire, chemin)

for dossier in [RACINE, RACINE / "predictions_validation", RACINE / "figures"]:
    dossier.mkdir(parents=True, exist_ok=True)
print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__)
print("Résultats dans :", RACINE)

## 1. Préparer les images v3 et vérifier ce qu'elles contiennent encore

Les 4 753 images (apprentissage et validation) sont préparées une fois et gardées en mémoire. Une figure compare la préparation v2 et la préparation v3 sur les trois premières radios de chaque classe de l'apprentissage.

Ensuite, on vérifie sur les **entrées v3 elles-mêmes** (ce que le réseau voit) si des statistiques globales séparent encore les classes :

- **intensité** : moyenne, écart-type et percentiles. Après égalisation, ces statistiques devraient être presque identiques d'une image à l'autre, et leur AUC proche de 0,5 ;
- **netteté** : écart moyen entre pixels voisins. C'est un indice possible de la résolution d'origine, qui n'est pas supprimé par v3. Une netteté qui sépare les classes serait un raccourci résiduel, mais elle peut aussi refléter de vraies opacités, qui estompent les détails.

Pour mémoire, le format d'origine (largeur, hauteur) sépare toujours les classes : cette information existe dans les métadonnées. La question est de savoir si le réseau v3 peut encore la deviner ; la section 3 le mesure.

In [ ]:
def preparer_v3(image):
    """Niveaux de gris, étirement en 224 × 224 sans bandes noires, égalisation d'histogramme."""
    gris = image.convert("L")
    etiree = gris.resize((TAILLE, TAILLE), Image.Resampling.BILINEAR)
    return np.array(ImageOps.equalize(etiree), dtype=np.uint8)

def preparer_v2(image):
    """Préparation v2, utilisée ici seulement pour la figure de comparaison."""
    return np.array(ImageOps.pad(image.convert("L"), (TAILLE, TAILLE), method=Image.Resampling.BILINEAR,
                                 color=0, centering=(0.5, 0.5)), dtype=np.uint8)

def statistiques_entree(tableau_u8):
    x = tableau_u8.astype(np.float64) / 255.0
    p05, p50, p95 = np.percentile(x, [5, 50, 95])
    nettete = float(np.abs(np.diff(x, axis=0)).mean() + np.abs(np.diff(x, axis=1)).mean())
    return {"moyenne": float(x.mean()), "ecart_type": float(x.std()), "p05": float(p05), "p50": float(p50),
            "p95": float(p95), "nettete": nettete}

train_df = pd.read_csv(REPARTITION / "train.csv", dtype={"groupe": str, "empreinte": str})
val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
for nom, tableau, attendus in [("train", train_df, {"NORMAL": 1079, "PNEUMONIA": 2721}), ("val", val_df, {"NORMAL": 269, "PNEUMONIA": 684})]:
    exiger({"chemin_relatif", "groupe", "empreinte", "classe"}.issubset(tableau.columns), f"Colonnes absentes : {nom}.")
    exiger(tableau["classe"].value_counts().to_dict() == attendus, f"Effectifs inattendus : {nom}.")
    exiger(tableau["chemin_relatif"].is_unique, f"Chemins dupliqués : {nom}.")
for colonne in ["groupe", "empreinte", "chemin_relatif"]:
    exiger(set(train_df[colonne]).isdisjoint(val_df[colonne]), f"Chevauchement entre apprentissage et validation : {colonne}.")

def preparer_ensemble(tableau, ensemble):
    images = np.empty((len(tableau), TAILLE, TAILLE), dtype=np.uint8)
    lignes = []
    empreinte = hashlib.sha256()
    for i, ligne in enumerate(tqdm(tableau.to_dict("records"), desc=f"Préparation v3 : {ensemble}")):
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            largeur, hauteur = image.size
            images[i] = preparer_v3(image)
        empreinte.update(ligne["chemin_relatif"].encode("utf-8") + b"\0")
        empreinte.update(images[i].tobytes())
        lignes.append({"ensemble": ensemble, "chemin_relatif": ligne["chemin_relatif"], "groupe": ligne["groupe"],
                       "classe": ligne["classe"], "largeur": largeur, "hauteur": hauteur,
                       "rapport_largeur_hauteur": largeur / hauteur, **statistiques_entree(images[i])})
    return images, pd.DataFrame(lignes), empreinte.hexdigest()

IMAGES_TRAIN, entrees_train, SHA_TRAIN_V3 = preparer_ensemble(train_df, "train")
IMAGES_VAL, entrees_val, SHA_VAL_V3 = preparer_ensemble(val_df, "val")
ENTREES = pd.concat([entrees_train, entrees_val], ignore_index=True)
Y_TRAIN = train_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
Y_VAL = val_df["classe"].map(CLASSES).to_numpy(dtype=np.int64)
ecrire_csv(ENTREES, RACINE / "caracteristiques_entrees_v3.csv")
print("Images préparées :", len(IMAGES_TRAIN), "pour l'apprentissage et", len(IMAGES_VAL), "pour la validation.",
      f"Mémoire utilisée : {(IMAGES_TRAIN.nbytes + IMAGES_VAL.nbytes) / 1e6:.0f} Mo.")

# Figure : les mêmes radios avec la préparation v2 et la préparation v3.
exemples = pd.concat([train_df[train_df["classe"] == "NORMAL"].head(3), train_df[train_df["classe"] == "PNEUMONIA"].head(3)])
fig, axes = plt.subplots(2, len(exemples), figsize=(3 * len(exemples), 7.0), constrained_layout=True)
for colonne, ligne in enumerate(exemples.to_dict("records")):
    with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
        largeur, hauteur = image.size
        apercus = [(preparer_v2(image), f"{ligne['classe']} · {largeur} × {hauteur} px\nv2 : bandes noires"),
                   (preparer_v3(image), "v3 : étirée et égalisée")]
    for rang, (tableau, titre) in enumerate(apercus):
        axes[rang, colonne].imshow(tableau, cmap="gray", vmin=0, vmax=255)
        axes[rang, colonne].set_title(titre, fontsize=9)
        axes[rang, colonne].axis("off")
fig.suptitle("Mêmes radios d'apprentissage : préparation v2 (en haut) et v3 (en bas)")
enregistrer_figure(fig, RACINE / "figures" / "preparation_v2_v3.png")
plt.show()
plt.close(fig)

In [ ]:
VARIABLES_ENTREE = ["moyenne", "ecart_type", "p05", "p50", "p95", "nettete"]

def auc_orientee(y, valeurs):
    """AUC de séparation (0,5 = aucune, 1 = parfaite) et classe qui a les valeurs les plus élevées."""
    if np.ptp(valeurs) == 0:
        return 0.5, "aucune"
    auc = roc_auc_score(y, valeurs)
    return max(auc, 1 - auc), ("PNEUMONIA" if auc > 0.5 else ("NORMAL" if auc < 0.5 else "aucune"))

def modele_logistique(variables):
    apprentissage = ENTREES[ENTREES["ensemble"] == "train"]
    validation = ENTREES[ENTREES["ensemble"] == "val"]
    modele = make_pipeline(StandardScaler(), LogisticRegression(class_weight="balanced", max_iter=5000))
    modele.fit(apprentissage[variables].to_numpy(dtype=float), (apprentissage["classe"] == "PNEUMONIA").to_numpy(dtype=int))
    probabilites = modele.predict_proba(validation[variables].to_numpy(dtype=float))[:, 1]
    y = (validation["classe"] == "PNEUMONIA").to_numpy(dtype=int)
    return {"auc_validation": roc_auc_score(y, probabilites),
            "f1_macro_validation": f1_score(y, (probabilites >= 0.5).astype(int), average="macro")}

lignes = []
for variable in VARIABLES_ENTREE:
    ligne = {"variable": variable}
    apprentissage = ENTREES[ENTREES["ensemble"] == "train"]
    ligne["moyenne_NORMAL"] = apprentissage.loc[apprentissage["classe"] == "NORMAL", variable].mean()
    ligne["moyenne_PNEUMONIA"] = apprentissage.loc[apprentissage["classe"] == "PNEUMONIA", variable].mean()
    for ensemble in ["train", "val"]:
        tableau = ENTREES[ENTREES["ensemble"] == ensemble]
        auc, sens = auc_orientee((tableau["classe"] == "PNEUMONIA").to_numpy(dtype=int), tableau[variable].to_numpy(dtype=float))
        ligne[f"auc_separation_{ensemble}"] = auc
        ligne[f"valeurs_plus_elevees_chez_{ensemble}"] = sens
    lignes.append(ligne)
DIAGNOSTIC_ENTREES = pd.DataFrame(lignes)
MODELES_SIMPLES = pd.DataFrame([
    {"informations_utilisees": "Intensité de l'entrée v3 (moyenne, écart-type, percentiles)",
     **modele_logistique(["moyenne", "ecart_type", "p05", "p50", "p95"])},
    {"informations_utilisees": "Netteté de l'entrée v3", **modele_logistique(["nettete"])},
    {"informations_utilisees": "Format d'origine (largeur, hauteur, rapport) : métadonnées, non visibles directement en v3",
     **modele_logistique(["largeur", "hauteur", "rapport_largeur_hauteur"])},
])
ecrire_csv(DIAGNOSTIC_ENTREES, RACINE / "diagnostic_entrees_v3.csv")
ecrire_csv(MODELES_SIMPLES, RACINE / "diagnostic_modeles_simples_v3.csv")
print("Statistiques des entrées v3 : moyennes sur l'apprentissage et pouvoir séparateur (AUC : 0,5 = aucun, 1 = parfait)")
display(DIAGNOSTIC_ENTREES.round(4))
print("Petits modèles logistiques entraînés sur l'apprentissage, évalués sur la validation :")
display(MODELES_SIMPLES.round(4))
print("Rappel du notebook 11 (entrées v2) : intensité seule AUC 0,908 ; format seul AUC 0,935.")

## 2. Entraîner les six modèles v3

Le code d'entraînement est celui du notebook 07 v2 : mêmes graines, même ordre d'initialisation, mêmes sauvegardes à chaque époque et même reprise après interruption. Deux différences seulement : les images viennent de la mémoire au lieu d'être relues sur le disque, et les écritures réessaient si OneDrive verrouille un fichier.

Les sauvegardes sont écrites dans `v3_sans_raccourci_format/<modèle>/graine_<graine>/configuration_<indice>/`.

In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # Connexion résiduelle, identique aux notebooks 06 v2 à 11.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))

ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
with torch.random.fork_rng(devices=[]):
    lot_controle = torch.from_numpy(IMAGES_TRAIN[:LOT].astype(np.float32) / 255.0).unsqueeze(1)
    for nom, (classe, _, nombre_parametres) in ARCHITECTURES.items():
        verification = classe().eval()
        with torch.no_grad():
            scores = verification(lot_controle)
        exiger(tuple(scores.shape) == (LOT, 2) and torch.isfinite(scores).all().item(), f"Sortie inattendue : {nom}.")
        exiger(sum(p.numel() for p in verification.parameters()) == nombre_parametres, f"Nombre de paramètres inattendu : {nom}.")
    del verification, lot_controle
print("Architectures vérifiées : 89 026 paramètres (classique) et 89 636 (CBAM résiduel), identiques à v2.")

In [ ]:
class Entrees(Dataset):
    """Images v3 déjà préparées, gardées en mémoire."""
    def __init__(self, images, etiquettes):
        self.images, self.etiquettes = images, etiquettes
    def __len__(self):
        return len(self.images)
    def __getitem__(self, i):
        x = torch.from_numpy(self.images[i].astype(np.float32) / 255.0).unsqueeze(0)
        return x, int(self.etiquettes[i])

TRAIN, VAL = Entrees(IMAGES_TRAIN, Y_TRAIN), Entrees(IMAGES_VAL, Y_VAL)
effectifs = train_df["classe"].value_counts()
POIDS = torch.tensor([len(train_df) / (2 * effectifs[c]) for c in ["NORMAL", "PNEUMONIA"]], dtype=torch.float32)
SHA_CSV = {s: empreinte_fichier(REPARTITION / f"{s}.csv") for s in ["train", "val"]}
print("Poids NORMAL / PNEUMONIA :", POIDS.tolist())

def construire_protocole(architecture, graine, lr, dropout):
    return {
        "version": "v3_sans_raccourci_format", "architecture": architecture, "graine": graine,
        "lot": LOT, "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
        "critere": "val_f1_macro", "amelioration": "strictement superieure",
        "grille": [[lr, dropout]], "optimiseur": "Adam", "weight_decay": 0.0,
        "pretraitement": PRETRAITEMENT_V3, "classes": CLASSES, "poids_classes": POIDS.tolist(),
        "sha256_csv": SHA_CSV, "sha256_entrees_v3": {"train": SHA_TRAIN_V3, "val": SHA_VAL_V3},
        "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
        "limite": "Reglages choisis avec la preparation v2, non reoptimises; test deja consulte dans le projet; indices residuels possibles (deformation, resolution, marqueurs).",
    }

def une_epoque(modele, chargeur, critere, optimiseur=None):
    apprentissage = optimiseur is not None
    modele.train(apprentissage)
    pertes, poids_total = 0.0, 0.0
    reels, predits = [], []
    for images, etiquettes in tqdm(chargeur, desc="Apprentissage" if apprentissage else "Validation", leave=False):
        images, etiquettes = images.to(APPAREIL), etiquettes.to(APPAREIL)
        if apprentissage: optimiseur.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(apprentissage):
            scores = modele(images)
            perte = critere(scores, etiquettes)
            if not torch.isfinite(perte).item(): raise RuntimeError("Perte non finie : arrêter et examiner cette configuration.")
            if apprentissage:
                perte.backward()
                optimiseur.step()
        poids = critere.weight[etiquettes].sum().item()
        pertes += perte.item()*poids
        poids_total += poids
        reels.extend(etiquettes.cpu().tolist())
        predits.extend(scores.detach().argmax(1).cpu().tolist())
    cm = confusion_matrix(reels, predits, labels=[0,1])
    tn,fp,fn,tp = cm.ravel()
    mesures = {
        "perte": float(pertes/poids_total), "exactitude": float(accuracy_score(reels,predits)),
        "precision_pneumonie": float(precision_score(reels,predits,zero_division=0)),
        "rappel_pneumonie": float(recall_score(reels,predits,zero_division=0)),
        "specificite_NORMAL": float(tn/(tn+fp)),
        "f1_pneumonie": float(f1_score(reels,predits,zero_division=0)),
        "f1_macro": float(f1_score(reels,predits,labels=[0,1],average="macro",zero_division=0))
    }
    return mesures, cm.tolist()

def entrainer(nom, graine):
    classe, architecture, _ = ARCHITECTURES[nom]
    reglages = CONFIGURATIONS[nom]
    configuration = dict(reglages)
    dossier = RACINE / nom / f"graine_{graine}" / f"configuration_{reglages['indice']:02d}"
    dossier.mkdir(parents=True, exist_ok=True)
    protocole = construire_protocole(architecture, graine, reglages["lr"], reglages["dropout"])
    fichier_protocole = dossier / "protocole.json"
    if fichier_protocole.exists():
        exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == protocole,
               f"Protocole de reprise différent pour {nom}, graine {graine} : même noyau, mêmes versions et mêmes images nécessaires.")
    else:
        ecrire_json(protocole, fichier_protocole)
    # Même séquence que les notebooks 05 à 07 : poids initiaux et ordre des lots identiques à v2 pour une même graine.
    torch.manual_seed(graine)
    modele = classe(reglages["dropout"]).to(APPAREIL)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=reglages["lr"])
    critere = nn.CrossEntropyLoss(weight=POIDS.to(APPAREIL))
    generateur = torch.Generator().manual_seed(graine)
    chargeur_train = DataLoader(TRAIN, batch_size=LOT, shuffle=True, generator=generateur, num_workers=0)
    chargeur_val = DataLoader(VAL, batch_size=LOT, shuffle=False, num_workers=0)
    torch.manual_seed(graine)
    historique = []
    def sauvegarder(epoque, matrice=None):
        sauver_torch({"epoque": epoque, "modele": modele.state_dict(), "optimiseur": optimiseur.state_dict(),
                      "historique": historique, "configuration": configuration, "protocole": protocole,
                      "etat_torch": torch.get_rng_state(), "etat_generateur": generateur.get_state(),
                      "matrice_validation": matrice}, dossier / f"epoque_{epoque:03d}.pt")
    disponibles = sorted(dossier.glob("epoque_[0-9][0-9][0-9].pt"))
    if disponibles:
        etat = torch.load(disponibles[-1], map_location="cpu", weights_only=True)
        exiger(etat["configuration"] == configuration and etat["protocole"] == protocole, "Sauvegarde de reprise incompatible.")
        modele.load_state_dict(etat["modele"], strict=True)
        optimiseur.load_state_dict(etat["optimiseur"])
        historique = etat["historique"]
        exiger(etat["epoque"] == len(historique), "Historique de reprise incomplet.")
        torch.set_rng_state(etat["etat_torch"])
        generateur.set_state(etat["etat_generateur"])
        print("Reprise après l'époque", etat["epoque"])
    else:
        sauvegarder(0)
    while len(historique) < MAX_EPOQUES:
        if historique:
            meilleur = max(historique, key=lambda r: r["val_f1_macro"])
            if len(historique) - meilleur["epoque"] >= PATIENCE:
                break
        epoque = len(historique) + 1
        debut = time.perf_counter()
        tr, _ = une_epoque(modele, chargeur_train, critere, optimiseur)
        va, cm = une_epoque(modele, chargeur_val, critere)
        ligne = {"epoque": epoque, "duree_secondes": float(time.perf_counter() - debut)}
        ligne.update({f"train_{k}": v for k, v in tr.items()})
        ligne.update({f"val_{k}": v for k, v in va.items()})
        historique.append(ligne)
        sauvegarder(epoque, cm)
        ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
        print(f"Époque {epoque:02d} | F1 macro val {va['f1_macro']:.4f} | rappel {va['rappel_pneumonie']:.4f} | "
              f"spécificité {va['specificite_NORMAL']:.4f} | {ligne['duree_secondes'] / 60:.1f} min")
    meilleur = max(historique, key=lambda r: r["val_f1_macro"])
    source = dossier / f"epoque_{meilleur['epoque']:03d}.pt"
    temporaire = dossier / "meilleur.tmp"
    shutil.copy2(source, temporaire)
    remplacer(temporaire, dossier / "meilleur.pt")
    ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
    resume = {**configuration, "epoque_retenue": meilleur["epoque"], "epoque_finale": len(historique),
              "arret": "patience" if len(historique) - meilleur["epoque"] >= PATIENCE else "maximum_epoques",
              "duree_minutes": sum(r["duree_secondes"] for r in historique) / 60,
              "sauvegarde": str(dossier / "meilleur.pt"),
              **{k: v for k, v in meilleur.items() if k.startswith("val_")}}
    ecrire_json(resume, dossier / "resume.json")
    return resume

print("Fonctions prêtes. Aucun entraînement lancé jusqu'ici.")

### Lancer les six entraînements

**Cette cellule dure plusieurs heures.** Le tableau `resultats_v3.csv` est mis à jour après chaque modèle. Pour faire une pause, interrompre la cellule ; pour reprendre, redémarrer le noyau et réexécuter toutes les cellules dans l'ordre. Les modèles terminés sont reconnus en quelques secondes.

In [ ]:
resumes = []
for nom in ["CNN_classique", "CNN_CBAM"]:
    for graine in GRAINES:
        reglages = CONFIGURATIONS[nom]
        print(f"\n===== v3 | {nom} | graine {graine} | LR={reglages['lr']}, Dropout={reglages['dropout']} =====")
        resumes.append({"modele": nom, "graine": graine, **entrainer(nom, graine)})
        RESULTATS_V3 = pd.DataFrame(resumes)
        ecrire_csv(RESULTATS_V3, RACINE / "resultats_v3.csv")
        display(RESULTATS_V3[["modele", "graine", "epoque_retenue", "epoque_finale", "arret", "val_f1_macro",
                              "val_rappel_pneumonie", "val_specificite_NORMAL"]].round(4))
exiger(len(RESULTATS_V3) == 6, "Certains entraînements sont incomplets.")
print("Les six entraînements v3 sont terminés.")

## 3. Comparer v2 et v3 sur la validation

Les six modèles v3 sont rechargés depuis `meilleur.pt` et réévalués : leur F1 doit retrouver exactement celui de l'entraînement. Leurs scores sont enregistrés dans `predictions_validation/`. Les scores v2 sont repris des prédictions déjà enregistrées par les notebooks 08 v2 (sans augmentation) et 10 v2 (avec augmentation).

Trois comparaisons, toutes appariées par architecture et par graine :

1. **Performances** : F1 macro, AUC, rappel et spécificité des 18 modèles.
2. **Dépendance au format** : au sein de chaque classe, corrélation de Spearman entre le rapport largeur/hauteur de l'image d'origine et le score PNEUMONIA (logit PNEUMONIA − logit NORMAL). Proche de 0 : le score ne dépend pas du format.
3. **Erreurs selon le format** : taux de faux négatifs parmi les radios PNEUMONIA et de faux positifs parmi les radios NORMAL, par tiers de format (tiers calculés sur la validation, séparément pour chaque classe).

In [ ]:
def logits_validation(modele):
    modele.eval()
    sorties = []
    with torch.no_grad():
        for images, _ in DataLoader(VAL, batch_size=LOT, shuffle=False, num_workers=0):
            sorties.append(modele(images.to(APPAREIL)).cpu().numpy())
    return np.concatenate(sorties).astype(np.float64)

def predictions_v2(variante, nom, graine):
    if variante == "sans":
        chemin = DOSSIER_08 / "predictions" / f"{nom}_graine_{graine}__originale.csv"
    else:
        chemin = DOSSIER_10 / "predictions" / f"avec__{nom}_graine_{graine}__originale.csv"
    exiger(chemin.is_file(), f"Prédictions v2 introuvables : {chemin}")
    tableau = pd.read_csv(chemin, dtype={"groupe": str})
    exiger(tableau["chemin_relatif"].tolist() == val_df["chemin_relatif"].tolist(), f"Ordre des images différent : {chemin}")
    return tableau[["logit_NORMAL", "logit_PNEUMONIA"]].to_numpy(dtype=np.float64)

LOGITS = {}
for resume in RESULTATS_V3.to_dict("records"):
    nom, graine = resume["modele"], int(resume["graine"])
    etat = torch.load(Path(resume["sauvegarde"]), map_location="cpu", weights_only=True)
    exiger(etat["protocole"]["pretraitement"] == PRETRAITEMENT_V3 and etat["configuration"] == CONFIGURATIONS[nom],
           f"Sauvegarde v3 inattendue : {nom}, graine {graine}.")
    with torch.random.fork_rng(devices=[]):
        modele = ARCHITECTURES[nom][0](dropout=CONFIGURATIONS[nom]["dropout"]).to(APPAREIL)
        modele.load_state_dict(etat["modele"], strict=True)
    logits = logits_validation(modele)
    f1_recalcule = f1_score(Y_VAL, (logits[:, 1] > logits[:, 0]).astype(int), labels=[0, 1], average="macro", zero_division=0)
    exiger(abs(f1_recalcule - float(resume["val_f1_macro"])) < 1e-10, f"F1 recalculé différent pour {nom}, graine {graine}.")
    LOGITS[("v3 sans raccourci", nom, graine)] = logits
    tableau = val_df[["chemin_relatif", "groupe"]].copy()
    tableau["classe_vraie"] = Y_VAL
    tableau["prediction"] = (logits[:, 1] > logits[:, 0]).astype(int)
    tableau["logit_NORMAL"], tableau["logit_PNEUMONIA"] = logits[:, 0], logits[:, 1]
    tableau["modele"], tableau["graine"], tableau["epoque"] = nom, graine, int(etat["epoque"])
    ecrire_csv(tableau, RACINE / "predictions_validation" / f"{nom}_graine_{graine}.csv")
for variante, version in [("sans", "v2 sans augmentation"), ("avec", "v2 avec augmentation")]:
    for nom in ARCHITECTURES:
        for graine in GRAINES:
            LOGITS[(version, nom, graine)] = predictions_v2(variante, nom, graine)
exiger(len(LOGITS) == 18, "Il faut les scores des 18 modèles.")
print("Scores de validation réunis pour les 18 modèles (v3 réévalués, v2 repris des notebooks 08 v2 et 10 v2).")

In [ ]:
METRIQUES = ["f1_macro", "auc_roc", "rappel_pneumonie", "specificite_NORMAL", "exactitude"]

def mesures(y, logits):
    prediction = (logits[:, 1] > logits[:, 0]).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()
    return {"f1_macro": f1_score(y, prediction, labels=[0, 1], average="macro", zero_division=0),
            "auc_roc": roc_auc_score(y, logits[:, 1] - logits[:, 0]),
            "rappel_pneumonie": tp / (tp + fn), "specificite_NORMAL": tn / (tn + fp),
            "exactitude": (tn + tp) / len(y), "faux_positifs": int(fp), "faux_negatifs": int(fn)}

MESURES_VAL = pd.DataFrame([{"version": v, "modele": m, "graine": g, **mesures(Y_VAL, LOGITS[(v, m, g)])}
                            for v in VERSIONS_MODELES for m in ARCHITECTURES for g in GRAINES])
SYNTHESE_VAL = MESURES_VAL.groupby(["version", "modele"], sort=False)[METRIQUES + ["faux_positifs", "faux_negatifs"]].agg(["mean", "min", "max"])
SYNTHESE_VAL.columns = [f"{mesure}_{statistique}" for mesure, statistique in SYNTHESE_VAL.columns]
SYNTHESE_VAL = SYNTHESE_VAL.reset_index()

def ecarts(version_a, modele_a, version_b, modele_b, libelle):
    lignes = []
    for graine in GRAINES:
        a = MESURES_VAL[(MESURES_VAL["version"] == version_a) & (MESURES_VAL["modele"] == modele_a) & (MESURES_VAL["graine"] == graine)].iloc[0]
        b = MESURES_VAL[(MESURES_VAL["version"] == version_b) & (MESURES_VAL["modele"] == modele_b) & (MESURES_VAL["graine"] == graine)].iloc[0]
        lignes.append({"comparaison": libelle, "graine": graine, **{f"ecart_{m}": a[m] - b[m] for m in METRIQUES}})
    return lignes

lignes = []
for nom, libelle in [("CNN_classique", "classique"), ("CNN_CBAM", "CBAM")]:
    lignes += ecarts("v3 sans raccourci", nom, "v2 sans augmentation", nom, f"{libelle} : v3 − v2 sans augmentation")
for version in VERSIONS_MODELES:
    lignes += ecarts(version, "CNN_CBAM", version, "CNN_classique", f"{version} : CBAM − classique")
ECARTS = pd.DataFrame(lignes)
RESUME_ECARTS = ECARTS.groupby("comparaison", sort=False)[[f"ecart_{m}" for m in METRIQUES]].agg(["mean", "min", "max"])
RESUME_ECARTS.columns = [f"{mesure}_{statistique}" for mesure, statistique in RESUME_ECARTS.columns]
RESUME_ECARTS = RESUME_ECARTS.reset_index()
RESUME_ECARTS["graines_ou_ecart_F1_positif"] = ECARTS.groupby("comparaison", sort=False)["ecart_f1_macro"].apply(lambda s: int((s > 0).sum())).to_numpy()

format_val = ENTREES[ENTREES["ensemble"] == "val"].set_index("chemin_relatif").loc[val_df["chemin_relatif"]]
RAPPORT = format_val["rapport_largeur_hauteur"].to_numpy(dtype=float)
HAUTEUR = format_val["hauteur"].to_numpy(dtype=float)
lignes = []
for (version, nom, graine), logits in LOGITS.items():
    score = logits[:, 1] - logits[:, 0]
    for classe in [0, 1]:
        masque = Y_VAL == classe
        lignes.append({"version": version, "modele": nom, "graine": graine, "classe_vraie": "NORMAL" if classe == 0 else "PNEUMONIA",
                       "rho_rapport_score": float(spearmanr(RAPPORT[masque], score[masque])[0]),
                       "rho_hauteur_score": float(spearmanr(HAUTEUR[masque], score[masque])[0])})
DEPENDANCE = pd.DataFrame(lignes)
RESUME_DEPENDANCE = DEPENDANCE.groupby(["classe_vraie", "version", "modele"], sort=False)[["rho_rapport_score", "rho_hauteur_score"]].agg(["mean", "min", "max"])
RESUME_DEPENDANCE.columns = [f"{mesure}_{statistique}" for mesure, statistique in RESUME_DEPENDANCE.columns]
RESUME_DEPENDANCE = RESUME_DEPENDANCE.reset_index()

TIERS = {}
for classe in [0, 1]:
    bornes = np.quantile(RAPPORT[Y_VAL == classe], [1 / 3, 2 / 3])
    TIERS[classe] = np.digitize(RAPPORT, bornes)
LIBELLES_TIERS = ["format le plus étroit", "format intermédiaire", "format le plus large"]
lignes = []
for (version, nom, graine), logits in LOGITS.items():
    prediction = (logits[:, 1] > logits[:, 0]).astype(int)
    for classe, erreur in [(0, "faux positifs (radios NORMAL)"), (1, "faux négatifs (radios PNEUMONIA)")]:
        for rang, libelle in enumerate(LIBELLES_TIERS):
            masque = (Y_VAL == classe) & (TIERS[classe] == rang)
            lignes.append({"version": version, "modele": nom, "graine": graine, "erreur": erreur, "tiers_format": libelle,
                           "n_images": int(masque.sum()), "taux_erreur_pct": 100 * float((prediction[masque] != classe).mean())})
ERREURS_FORMAT = pd.DataFrame(lignes)
RESUME_ERREURS = (ERREURS_FORMAT.groupby(["erreur", "version", "modele", "tiers_format"], sort=False)["taux_erreur_pct"].mean()
                  .unstack("tiers_format")[LIBELLES_TIERS].reset_index())

ecrire_csv(MESURES_VAL, RACINE / "mesures_validation_v2_v3.csv")
ecrire_csv(SYNTHESE_VAL, RACINE / "synthese_validation_v2_v3.csv")
ecrire_csv(ECARTS, RACINE / "ecarts_apparies_validation.csv")
ecrire_csv(RESUME_ECARTS, RACINE / "resume_ecarts_validation.csv")
ecrire_csv(DEPENDANCE, RACINE / "dependance_format_validation.csv")
ecrire_csv(RESUME_DEPENDANCE, RACINE / "resume_dependance_format_validation.csv")
ecrire_csv(ERREURS_FORMAT, RACINE / "erreurs_par_format_validation.csv")

print("Performances sur la validation (moyenne, minimum et maximum sur 3 graines) :")
display(SYNTHESE_VAL[["version", "modele", "f1_macro_mean", "f1_macro_min", "f1_macro_max", "auc_roc_mean",
                      "rappel_pneumonie_mean", "specificite_NORMAL_mean"]].round(4))
print("Écarts appariés par graine :")
display(RESUME_ECARTS[["comparaison", "ecart_f1_macro_mean", "ecart_f1_macro_min", "ecart_f1_macro_max",
                       "graines_ou_ecart_F1_positif", "ecart_auc_roc_mean", "ecart_specificite_NORMAL_mean"]].round(4))
print("Dépendance au format : corrélation de Spearman entre le rapport largeur/hauteur d'origine et le score PNEUMONIA, au sein de chaque classe :")
display(RESUME_DEPENDANCE[["classe_vraie", "version", "modele", "rho_rapport_score_mean", "rho_rapport_score_min",
                           "rho_rapport_score_max", "rho_hauteur_score_mean"]].round(3))
print("Taux d'erreur (%) par tiers de format, moyenne sur 3 graines :")
display(RESUME_ERREURS.round(1))

In [ ]:
# Figure : F1 macro et dépendance au format, par version et par architecture (points = graines, trait = moyenne).
positions_modeles = {"CNN_classique": 0, "CNN_CBAM": 1}
decalages = {"v2 sans augmentation": -0.22, "v2 avec augmentation": 0.0, "v3 sans raccourci": 0.22}
panneaux = [
    ("F1 macro sur la validation", "F1 macro", lambda v, m: MESURES_VAL[(MESURES_VAL["version"] == v) & (MESURES_VAL["modele"] == m)]["f1_macro"]),
    ("Dépendance au format parmi les radios PNEUMONIA", "Corrélation de Spearman\nformat d'origine → score PNEUMONIA",
     lambda v, m: DEPENDANCE[(DEPENDANCE["version"] == v) & (DEPENDANCE["modele"] == m) & (DEPENDANCE["classe_vraie"] == "PNEUMONIA")]["rho_rapport_score"]),
]
fig, axes = plt.subplots(1, 2, figsize=(14, 5.2))
for axe, (titre, etiquette, valeurs) in zip(axes, panneaux):
    for version in VERSIONS_MODELES:
        for nom, x in positions_modeles.items():
            y = valeurs(version, nom).to_numpy(dtype=float)
            xs = np.full(len(y), x + decalages[version])
            axe.scatter(xs, y, s=64, color=COULEURS[version], edgecolors="white", linewidths=1.5, zorder=3,
                        label=version if nom == "CNN_classique" else None)
            axe.hlines(y.mean(), xs[0] - 0.08, xs[0] + 0.08, color=COULEURS[version], linewidth=2, zorder=2)
    axe.set_xticks(list(positions_modeles.values()), ["CNN classique", "CNN + CBAM"])
    axe.set_xlim(-0.5, 1.5)
    axe.set_title(titre, fontsize=11)
    axe.set_ylabel(etiquette)
    axe.grid(axis="y", color="#e5e5e5", linewidth=0.8)
    axe.set_axisbelow(True)
    for cote in ["top", "right"]:
        axe.spines[cote].set_visible(False)
axes[1].axhline(0, color="#8a8a8a", linewidth=1)
axes[0].legend(fontsize=9, frameon=False, loc="best")
fig.suptitle("Validation : v2 et v3, à architecture et graine égales (points : graines ; trait : moyenne)")
fig.tight_layout(rect=(0, 0, 1, 0.94))
enregistrer_figure(fig, RACINE / "figures" / "comparaison_v2_v3_validation.png")
plt.show()
plt.close(fig)
print("Fichiers à transmettre :")
for nom_fichier in ["diagnostic_entrees_v3.csv", "diagnostic_modeles_simples_v3.csv", "resultats_v3.csv", "synthese_validation_v2_v3.csv",
                    "resume_ecarts_validation.csv", "resume_dependance_format_validation.csv", "erreurs_par_format_validation.csv"]:
    print(" -", RACINE / nom_fichier)
print("Figures :", RACINE / "figures")

## Lecture des résultats

- **Diagnostic des entrées :** après égalisation, l'intensité ne devrait plus séparer les classes (AUC proche de 0,5). Si la netteté les sépare nettement, c'est un indice résiduel lié à la résolution d'origine ; il faudra le mentionner dans les limites.
- **F1 de validation :** une baisse en v3 est attendue et ne signifie pas un modèle moins bon. Elle montre la part des performances v2 qui venait des raccourcis. La vraie mesure de la généralisation sera le test.
- **Dépendance au format :** une corrélation qui se rapproche de 0 en v3, et des faux négatifs répartis plus également entre les tiers de format, indiquent que le raccourci a été réduit. Une corrélation encore forte signifierait que le réseau devine le format autrement (déformation, texture).
- **CBAM contre classique en v3 :** c'est la comparaison la plus intéressante pour le mémoire, car elle se fait sans le raccourci principal.

**Suite :** le notebook 13 vérifiera les cartes Grad-CAM des modèles v3 (regardent-ils davantage les poumons ?) et testera directement l'effet des bandes noires sur les modèles v2. Le notebook 14 fera ensuite l'évaluation finale sur le test, une seule fois, avec les quatre hypothèses écrites en tête de ce notebook.